# Artificial Neural Network for NSL-KDD Intrusion Detection

## Overview

This notebook implements a **supervised ANN classifier** for multi-class network intrusion detection using the NSL-KDD dataset.

### Model Architecture
- Input layer → Dense(45, ReLU) → Dropout(0.2) → Dense(45, ReLU) → Dense(5, Softmax)
- Loss: `sparse_categorical_crossentropy`
- Optimizer: Adam

### Evaluation
- Confusion matrix and classification report
- K-Fold cross-validation (10 folds)
- GridSearchCV for hyperparameter tuning

## 1. Setup and Data Loading

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

from src.preprocessing import (
    load_nsl_kdd, encode_attack_categories, CATEGORY_NAMES,
)

DATA_DIR = os.path.join('..', 'data')
full, n_train = load_nsl_kdd(
    os.path.join(DATA_DIR, 'KDDTrain+.csv'),
    os.path.join(DATA_DIR, 'KDDtest.csv'),
)

full['outcome'] = encode_attack_categories(full)
del full['score']

print(f"Total samples: {full.shape[0]:,}")
print(f"Training: {n_train:,} | Test: {full.shape[0] - n_train:,}")

## 2. Feature Engineering and Preprocessing

One-hot encode categoricals, then normalize features using L2 normalization.

In [ ]:
full_encoded = pd.get_dummies(full, drop_first=True)
print(f"Features after encoding: {full_encoded.shape[1] - 1}")

X_train = np.array(full_encoded[:n_train].drop(columns=['outcome']))
y_train = np.array(full_encoded['outcome'][:n_train])
X_test = np.array(full_encoded[n_train:].drop(columns=['outcome']))
y_test = np.array(full_encoded['outcome'][n_train:])

X_train = normalize(X_train)
X_test = normalize(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 3. Build and Train the ANN

Two hidden layers with 45 neurons each, dropout for regularization, and softmax output for 5-class classification.

In [ ]:
import time
from src.ann_classifier.nsl_kdd_classifier import build_ann_model

model = build_ann_model(input_dim=X_train.shape[1])
model.summary()

start = time.time()
model.fit(X_train, y_train, batch_size=10, epochs=3)
print(f"\nTraining time: {time.time() - start:.2f} seconds")

## 4. Evaluation

### 4.1 Predictions and Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, precision_score,
    confusion_matrix, classification_report,
)

y_pred = model.predict(X_test).argmax(axis=-1)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='micro'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, average='micro'):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='micro'):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

### 4.2 K-Fold Cross Validation

10-fold cross-validation to estimate model generalization.

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score

def make_model():
    return build_ann_model(input_dim=X_train.shape[1])

keras_clf = KerasClassifier(model=make_model, batch_size=10, epochs=3, verbose=0)

# Cross-validation on training set
scores_train = cross_val_score(keras_clf, X_train, y_train, cv=10, n_jobs=1)
print(f"Train CV — Mean: {scores_train.mean():.4f}, Std: {scores_train.std():.4f}")

# Cross-validation on test set
scores_test = cross_val_score(keras_clf, X_test, y_test, cv=10, n_jobs=1)
print(f"Test CV  — Mean: {scores_test.mean():.4f}, Std: {scores_test.std():.4f}")

### 4.3 GridSearchCV — Hyperparameter Tuning

Search over batch size, epochs, and optimizer to find the best combination.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'batch_size': [10, 20],
    'epochs': [5, 10, 15],
    'optimizer': ['adam', 'rmsprop'],
}

def make_model_with_optimizer(optimizer='adam'):
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    m = Sequential([
        Dense(45, kernel_initializer='uniform', activation='relu', input_dim=X_train.shape[1]),
        Dropout(rate=0.2),
        Dense(45, kernel_initializer='uniform', activation='relu'),
        Dense(5, kernel_initializer='uniform', activation='softmax'),
    ])
    m.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

grid_clf = KerasClassifier(model=make_model_with_optimizer, verbose=0)
grid_search = GridSearchCV(grid_clf, param_grid, scoring='accuracy', n_jobs=1, cv=10)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.4f}")